# IPCC WGI reference regions (v4)

Reference map of the 58 IPCC AR6 WGI land/ocean regions with their acronyms and a latitude/longitude graticule, drawn in the same Robinson/Arial style as the other supplementary maps.

In [ ]:
# Supplementary figure — IPCC WGI reference regions (v4) reference map.
# Basemap and graticule match fig1 (Robinson projection; cartopy LAND/OCEAN/
# COASTLINE/BORDERS; Arial; dashed region outlines; labelled gridlines).
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd

# --- Nature style ---
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 16,
    "figure.dpi": 300,
    "axes.linewidth": 0.8,
})

# Run from this notebook's folder; the geojson sits next to it.
IPCC_PATH = Path("IPCC-WGI-reference-regions-v4.geojson")
ipcc = gpd.read_file(IPCC_PATH)
print(f"Loaded {len(ipcc)} IPCC reference regions")

fig = plt.figure(figsize=(14, 7), dpi=300)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
ax.set_global()

# --- Basemap (same cartopy features as fig1) ---
ax.set_facecolor("#FFFFFF")
ax.add_feature(cfeature.LAND, color="#CECECE", alpha=0.2)
ax.add_feature(cfeature.OCEAN, color="#FFFFFF", alpha=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.7, alpha=0.5)
ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.5, alpha=0.3)

# --- IPCC region boundaries (dashed outline, as in fig1) ---
for _, region_row in ipcc.iterrows():
    ax.add_geometries(
        [region_row["geometry"]], crs=ccrs.PlateCarree(),
        facecolor="none", edgecolor="#555555",
        linewidth=0.8, linestyle="--", zorder=1,
    )

# --- Region acronym labels at an interior representative point ---
for _, region_row in ipcc.iterrows():
    pt = region_row.geometry.representative_point()
    ax.text(
        pt.x, pt.y, region_row["Acronym"],
        transform=ccrs.PlateCarree(),
        ha="center", va="center", fontsize=8, fontweight="bold",
        color="#1a1a1a", zorder=11,
        path_effects=[pe.withStroke(linewidth=1.4, foreground="white")],
    )

# --- Graticule with latitude / longitude labels (size 16, as in fig1) ---
gl = ax.gridlines(
    crs=ccrs.PlateCarree(), draw_labels=True,
    linewidth=0.4, color="gray", alpha=0.5, linestyle="--",
    xlocs=range(-180, 181, 60), ylocs=range(-60, 91, 30),
)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {"size": 16, "color": "#404040"}
gl.ylabel_style = {"size": 16, "color": "#404040"}

# Clean frame, keep the graticule labels (as in fig1)
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
out = "suppfig_ipcc_reference_regions.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
print(f"Saved -> {out}")
plt.show()
